In [2]:
#from data_quality_test_management import *
from recipe_management import *
from dotenv import load_dotenv
import os
# Charger les variables du fichier .env
load_dotenv()
from rapidfuzz import fuzz, process
import re





In [3]:
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY_2')
MODEL_NAME = os.getenv('MODEL_NAME_2')

In [4]:
# jumelage des cohorts et creation des donnees vaec non geste et avec transformation de non geste en periode
df,df1 = process_files_smart(1,108,batch_size = 15)

CONFIGURATION DU TRAITEMENT
CPUs disponibles: 12
Processus utilisés: 6
Taille des lots: 15 fichiers
Cohortes à traiter: 106
Intervalle: [1 - 108]
Liste des cohortes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]...


LOT 1/8
Cohortes traitées dans ce lot: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Nombre de fichiers: 15
------------------------------------------------------------

✓ Lot 1 terminé!
  Temps écoulé: 29.83s (1.99s par fichier)
  Lignes ajoutées: 579,118 (final) + 144,783 (temporal)

⏱  Temps estimé restant: 3.0 minutes
  Progression: 14.2%
  Pause de 3 secondes...

LOT 2/8
Cohortes traitées dans ce lot: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
Nombre de fichiers: 15
------------------------------------------------------------

✓ Lot 2 terminé!
  Temps écoulé: 30.22s (2.01s par fichier)
  Lignes ajoutées: 577,864 (final) + 144,475 (temporal)

⏱  Temps estimé restant: 2.6 minutes
  Progression: 28.3%
  Pause de 3 secondes...

LOT 3/8
Cohortes traitées dans ce 

In [5]:
def data_preparation_3stages_1(file_path):
    
    with open(file_path, 'r') as f:
        save_data = json.load(f)
        results = save_data.get('results', save_data)
    
    # Charger les données de base et créer un dictionnaire
    recipes = pd.read_csv("recipes.csv", usecols=['id', 'title'])
    recipe_titles = dict(zip(recipes['id'], recipes['title']))
    
    # Pré-allouer les listes avec une estimation de taille
    estimated_size = len(results['variantes_principales']) * 5
    all_data = []
    temporal_data = []
    
    # Définir les configurations de variantes pour éviter la répétition de code
    variante_configs = [
        ('variantes_principales', None, 'principal', 'variante_principale'),
        ('variantes_secondaires', 'ingredient_variant', 'secondaire', 'variante_ingredients'),
        ('variantes_secondaires', 'permutation_1', 'secondaire', 'variante_permutation'),
        ('variantes_secondaires', 'permutation_2', 'secondaire', 'variante_permutation'),
    ]
    
    # Traitement de chaque recette
    for recipe_id in results['variantes_principales'].keys():
        recipe_title = recipe_titles.get(recipe_id, f"Recipe_{recipe_id}")
        
        # Traiter les variantes principales et secondaires
        for result_key, sub_key, type_val, type_2_val in variante_configs:
            if sub_key is None:
                variante = results[result_key].get(recipe_id, [])
            else:
                variante = results[result_key].get(recipe_id, {}).get(sub_key, [])
            
            if variante and variante != ["NA"]:
                all_data.append({
                    'id': recipe_id,
                    'title': recipe_title,
                    'actions': variante,
                    'type': type_val,
                    'type_2': type_2_val
                })
        
        # Variante ternaire (temporelle)
        variante_ternaire = results['variantes_ternaires'].get(recipe_id, [])
        if variante_ternaire and variante_ternaire != ["NA"]:
            temporal_data.append({
                'id': recipe_id,
                'title': recipe_title,
                'actions': variante_ternaire,
                'type': 'principal'
            })
    
    # Créer les DataFrames une seule fois
    final_df = pd.DataFrame(all_data)
    temporal_df = pd.DataFrame(temporal_data)

    #final_df.to_csv('dataset_finaux/dataset_final.csv', index=False)
    #temporal_df.to_csv('dataset_finaux/dataset_temporel_final.csv', index=False)
    
    return final_df, temporal_df
suffixes = ['X','50_to_80_','80_to_100_','101_to_108']
for suffix in suffixes:
    if suffix == 'X':
        file_path = f"recipes_variants_3stages/sauvegarde_final_3stages_cohort_{suffix}.json"
    else:
        file_path = f"recipes_variants_3stages/retry_na_results_cohorts_{suffix}.json"
    dat,dat1 = data_preparation_3stages_1(file_path)
    df = pd.concat([df,dat])
    df1 = pd.concat([df1,dat1])
df.to_csv('data/dataset_graph_recette.csv')
df1.to_csv('data/dataset_graph_recette_avec_temps.csv')   

In [6]:
df = pd.read_csv('data/dataset_graph_recette.csv')
data, data1 = data_cleaning_3stages(df)

→ Nettoyage de la colonne actions...

Nettoyage de la colonne 'actions'...
✅ Nettoyage terminé!
  ✓ Nettoyage terminé: 4,073,652 lignes
→ Netoyage et Normalisation des verbes...
🚀 Début du nettoyage optimisé...
   Dataset: 4,073,652 lignes
   Actions à supprimer: 1043
   Règles de normalisation: 478
   Seuil de similarité: 95

📋 Étape 1/3: Extraction des actions uniques...
   ✓ 5,186 actions uniques trouvées

🧹 Étape 2/3: Nettoyage des actions uniques...
   ✓ 4,304 actions conservées
   ✓ 882 actions supprimées (17.0%)

⚡ Étape 3/3: Application du mapping...

✅ Terminé en 56.48 secondes!
   Vitesse: 72,121 lignes/seconde

📊 Statistiques:
   Actions moyennes AVANT: 11.95
   Actions moyennes APRÈS: 11.86
   Réduction: 0.8%
  ✓ Netoyage et Normalisation terminée
→ Création d'une copie pour data_with_non_gesture...
  ✓ Copie créée: 4,073,652 lignes
→ Filtrage des gestes...
  ✓ Filtrage terminé
→ Suppression des doublons consécutifs...
  ✓ Doublons consécutifs supprimés
→ Suppression des do

In [15]:
recipes_df = pd.read_csv("recipes.csv")
recipe_instructions_df = pd.read_csv("recipe_instructions.csv")
recipe_ingredients_df = pd.read_csv("recipe_ingredients.csv")
graph_brut  = pd.read_csv("data/dataset_graph_recette.csv")
graph_nettoye =  pd.read_csv("data/final_dataset_cleaned_with_non_gestures.csv")
graph_geste_only  = pd.read_csv("data/final_dataset_cleaned_without_non_gestures.csv")


In [26]:
with open('actions_to_remove_final.txt', 'r', encoding='utf-8') as f:
   actions_to_remove = [line.strip() for line in f if line.strip()]

    # Importer normalization_dict depuis le fichier JSON
with open('normalization_dict_final.json', 'r', encoding='utf-8') as f:
    normalization_dict = json.load(f)

print(len(actions_to_remove))
print(len(normalization_dict))

2608
2296


In [ ]:


graph_brut["actions"] = graph_brut["actions"].apply(to_list_of_strings
 )
graph_brut =  clean_dataframe_optimized(
        data= graph_brut,
        actions_to_remove=actions_to_remove,
        normalization_dict=normalization_dict,
        similarity_threshold=95, 
        verbose=True
    )

graph_nettoye["actions"] = graph_nettoye["actions"].apply(to_list_of_strings
 )
graph_nettoye =  clean_dataframe_optimized(
        data= graph_nettoye,
        actions_to_remove=actions_to_remove,
        normalization_dict=normalization_dict,
        similarity_threshold=95,  
        verbose=True
    )

graph_geste_only["actions"] = graph_geste_only["actions"].apply(to_list_of_strings
 )
graph_geste_only =  clean_dataframe_optimized(
        data= graph_geste_only,
        actions_to_remove=actions_to_remove,
        normalization_dict=normalization_dict,
        similarity_threshold=95,  
        verbose=True
    )
    
    

🚀 Début du nettoyage optimisé...
   Dataset: 4,073,652 lignes
   Actions à supprimer: 2598
   Règles de normalisation: 2293
   Seuil de similarité: 95

📋 Étape 1/3: Extraction des actions uniques...
   ✓ 5,186 actions uniques trouvées

🧹 Étape 2/3: Nettoyage des actions uniques...
   ✓ 2,686 actions conservées
   ✓ 2,500 actions supprimées (48.2%)

⚡ Étape 3/3: Application du mapping...

✅ Terminé en 124.35 secondes!
   Vitesse: 32,761 lignes/seconde

📊 Statistiques:
   Actions moyennes AVANT: 11.95
   Actions moyennes APRÈS: 11.27
   Réduction: 5.7%
🚀 Début du nettoyage optimisé...
   Dataset: 2,749,491 lignes
   Actions à supprimer: 2598
   Règles de normalisation: 2293
   Seuil de similarité: 95

📋 Étape 1/3: Extraction des actions uniques...
   ✓ 3,614 actions uniques trouvées

🧹 Étape 2/3: Nettoyage des actions uniques...
   ✓ 2,089 actions conservées
   ✓ 1,525 actions supprimées (42.2%)

⚡ Étape 3/3: Application du mapping...

✅ Terminé en 138.60 secondes!
   Vitesse: 19,837 lig

In [27]:

graph_brut =  clean_dataframe_optimized(
        data= graph_brut,
        actions_to_remove=actions_to_remove,
        normalization_dict=normalization_dict,
        similarity_threshold=95, 
        verbose=True
    )


graph_nettoye =  clean_dataframe_optimized(
        data= graph_nettoye,
        actions_to_remove=actions_to_remove,
        normalization_dict=normalization_dict,
        similarity_threshold=95,  
        verbose=True
    )


graph_geste_only =  clean_dataframe_optimized(
        data= graph_geste_only,
        actions_to_remove=actions_to_remove,
        normalization_dict=normalization_dict,
        similarity_threshold=95,  
        verbose=True
    )

🚀 Début du nettoyage optimisé...
   Dataset: 4,073,652 lignes
   Actions à supprimer: 2608
   Règles de normalisation: 2296
   Seuil de similarité: 95

📋 Étape 1/3: Extraction des actions uniques...
   ✓ 722 actions uniques trouvées

🧹 Étape 2/3: Nettoyage des actions uniques...
   ✓ 704 actions conservées
   ✓ 18 actions supprimées (2.5%)

⚡ Étape 3/3: Application du mapping...

✅ Terminé en 257.86 secondes!
   Vitesse: 15,798 lignes/seconde

📊 Statistiques:
   Actions moyennes AVANT: 11.27
   Actions moyennes APRÈS: 11.27
   Réduction: 0.0%
🚀 Début du nettoyage optimisé...
   Dataset: 2,749,491 lignes
   Actions à supprimer: 2608
   Règles de normalisation: 2296
   Seuil de similarité: 95

📋 Étape 1/3: Extraction des actions uniques...
   ✓ 670 actions uniques trouvées

🧹 Étape 2/3: Nettoyage des actions uniques...
   ✓ 657 actions conservées
   ✓ 13 actions supprimées (1.9%)

⚡ Étape 3/3: Application du mapping...

✅ Terminé en 34.00 secondes!
   Vitesse: 80,872 lignes/seconde

📊 St

In [30]:
def to_list_of_strings(x):
    """Retourne une liste de strings propre pour une cellule actions."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
        except Exception:
            # si ce n'est pas une repr de liste, on en fait une liste avec la string
            return [x]
        else:
            return to_list_of_strings(parsed)
    if isinstance(x, list):
        # aplatir une couche si éléments sont listes
        flattened = []
        for e in x:
            if isinstance(e, list):
                flattened.extend(e)
            else:
                flattened.append(e)
        return [str(i) for i in flattened]
    # autre type scalaire
    return [str(x)]

import ast
from functools import lru_cache
from pathlib import Path
from typing import List, Union, Set


@lru_cache(maxsize=1)
def load_non_gesture_verbs(filepath: str = "data/non_gesture_verbs.txt") -> Set[str]:
    """
    Charge les verbes non-gestuels depuis un fichier texte.
    Utilise un cache pour éviter de relire le fichier à chaque appel.
    
    Args:
        filepath: Chemin vers le fichier contenant les verbes non-gestuels
        
    Returns:
        Set de verbes en minuscules pour une recherche O(1)
    """
    non_gesture_verbs = set()
    
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Le fichier '{filepath}' n'existe pas.")
    
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            # Ignorer les lignes vides et les commentaires
            if line and not line.startswith('#'):
                non_gesture_verbs.add(line.lower())
    
    return non_gesture_verbs


def filter_actions_1(
    actions: Union[str, List[str]], 
    non_gesture_filepath: str = "data/non_gesture_verbs.txt"
) -> List[str]:
    """
    Filtre les actions en supprimant les verbes non-gestuels.
    
    Args:
        actions: Liste d'actions (str ou List[str])
        non_gesture_filepath: Chemin vers le fichier des verbes non-gestuels
        
    Returns:
        Liste des actions gestuelles uniquement
    """
    # Charger les verbes non-gestuels (avec cache)
    non_gesture_verbs = load_non_gesture_verbs(non_gesture_filepath)
    
    # Convertir la chaîne en liste si nécessaire
    try:
        if isinstance(actions, str):
            actions_list = ast.literal_eval(actions)
        else:
            actions_list = actions
            
        if not isinstance(actions_list, list):
            return actions
            
    except (ValueError, SyntaxError):
        return actions
    
    # Filtrer en gardant seulement les verbes gestuels
    filtered_actions = [
        action for action in actions_list
        if action.lower() not in non_gesture_verbs
    ]
    
    return filtered_actions

In [31]:
graph_geste_only['actions'] = graph_geste_only['actions'].apply(filter_actions_1)

In [33]:
graph_brut.to_csv("data/dataset_graph_recette.csv")
graph_nettoye.to_csv("data/final_dataset_cleaned_with_non_gestures.csv")
graph_geste_only.to_csv("data/final_dataset_cleaned_without_non_gestures.csv")

In [32]:
# Vérifier verbes uniques après nettoyage
def extraire_verbes_uniques_from_col(df, col="actions"):
    s = set()
    for x in df[col].values:
        if not x:
            continue
        for tok in x:
            s.add(tok)
    return sorted(s)
verbes_apres = extraire_verbes_uniques_from_col(graph_geste_only, "actions")
print("nombre verbes après nettoyage:", len(verbes_apres))
print(verbes_apres[:100])   # premiers 100 pour contrôle
with open('echantillon_verbes_uniques.txt', 'w', encoding='utf-8') as f:
    for verbe in verbes_apres:
        f.write(verbe + '\n')

nombre verbes après nettoyage: 458
['', 'absorb', 'acidulate', 'add', 'adjust', 'aerate', 'agitate', 'alternate', 'amalgamate', 'anoint', 'apply', 'arrange', 'assemble', 'attach', 'bag', 'baggie', 'bar', 'bast', 'batten', 'batter', 'beard', 'beat', 'behead', 'blob', 'blot', 'bone', 'braid', 'break', 'bruise', 'brunoise', 'brush', 'buff', 'build', 'bundle', 'bung', 'burnish', 'butcher', 'butter', 'butterfly', 'can', 'cap', 'carve', 'cavity', 'center', 'chiffonade', 'chink', 'chip', 'chisel', 'choke', 'chop', 'chuck', 'chunk', 'claw', 'clean', 'clear', 'cleave', 'clip', 'clump', 'cluster', 'coat', 'coil', 'color', 'combine', 'concasse', 'concoct', 'condense', 'convert', 'core', 'cork', 'corner', 'cover', 'crack', 'cradle', 'cram', 'crank', 'crimp', 'crinkle', 'cross', 'crosscut', 'crosshatch', 'crumb', 'crumble', 'crumple', 'crush', 'crust', 'cube', 'curl', 'curve', 'cut', 'dab', 'damp', 'dampen', 'dash', 'decant', 'declump', 'decorate', 'degas', 'deglaze', 'degrease', 'devein']
